In [1]:
import os
os.system('pip install -q pyvi')

import re
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from pyvi import ViTokenizer
from tqdm import tqdm

# =====================================================
# 1) CONFIG
# =====================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

class Config:
    model_name = 'vinai/phobert-base'
    max_len = 180
    batch_size = 16
    epochs = 5
    lr = 2e-5
    weight_decay = 0.01
    n_folds = 5
    warmup_ratio = 0.1
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = Config()
print('Device:', cfg.device)

# =====================================================
# 2) LOAD DATA
# =====================================================
train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')

TEXT_COL = 'text' if 'text' in train_df.columns else train_df.columns[1]
LABEL_COL = 'sentiment'

# =====================================================
# 3) PREPROCESS
# =====================================================
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\bko\b', 'không', text)
    text = re.sub(r'\bok\b', 'ổn', text)
    return ViTokenizer.tokenize(text)

train_df[TEXT_COL] = train_df[TEXT_COL].fillna('').apply(clean_text)
test_df[TEXT_COL] = test_df[TEXT_COL].fillna('').apply(clean_text)

# =====================================================
# 4) TOKENIZER
# =====================================================
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=cfg.max_len,
            return_tensors='pt'
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0)
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# =====================================================
# 5) MODEL
# =====================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

class PhoBERTClassifier(nn.Module):
    def __init__(self, model_name, n_classes):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        x = self.dropout(cls)
        return self.fc(x)

# =====================================================
# 6) TRAIN FUNCTION
# =====================================================
def train_one_fold(train_idx, val_idx, fold):
    x_train = train_df.iloc[train_idx][TEXT_COL].tolist()
    y_train = train_df.iloc[train_idx][LABEL_COL].tolist()
    x_val = train_df.iloc[val_idx][TEXT_COL].tolist()
    y_val = train_df.iloc[val_idx][LABEL_COL].tolist()

    train_ds = SentimentDataset(x_train, y_train)
    val_ds = SentimentDataset(x_val, y_val)
    test_ds = SentimentDataset(test_df[TEXT_COL].tolist())

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size)

    model = PhoBERTClassifier(cfg.model_name, 3).to(cfg.device)

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(cfg.device)

    criterion = FocalLoss(alpha=class_weights, gamma=2)
    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    total_steps = len(train_loader) * cfg.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * cfg.warmup_ratio),
        num_training_steps=total_steps
    )

    best_f1 = 0
    best_test_probs = None

    for epoch in range(cfg.epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f'Fold {fold} Epoch {epoch+1}'):
            optimizer.zero_grad()
            ids = batch['input_ids'].to(cfg.device)
            mask = batch['attention_mask'].to(cfg.device)
            labels = batch['labels'].to(cfg.device)

            logits = model(ids, mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        preds, truths = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids = batch['input_ids'].to(cfg.device)
                mask = batch['attention_mask'].to(cfg.device)
                labels = batch['labels'].to(cfg.device)

                logits = model(ids, mask)
                pred = torch.argmax(logits, dim=1)
                preds.extend(pred.cpu().numpy())
                truths.extend(labels.cpu().numpy())

        f1 = f1_score(truths, preds, average='macro')
        print(f'Fold {fold} Epoch {epoch+1} Macro F1: {f1:.5f}')

        if f1 > best_f1:
            best_f1 = f1
            probs = []
            with torch.no_grad():
                for batch in test_loader:
                    ids = batch['input_ids'].to(cfg.device)
                    mask = batch['attention_mask'].to(cfg.device)
                    logits = model(ids, mask)
                    prob = torch.softmax(logits, dim=1)
                    probs.append(prob.cpu().numpy())
            best_test_probs = np.vstack(probs)

    return best_f1, best_test_probs

# =====================================================
# 7) K-FOLD TRAINING
# =====================================================
skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=42)
all_test_probs = []
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df[LABEL_COL]), 1):
    score, test_probs = train_one_fold(train_idx, val_idx, fold)
    fold_scores.append(score)
    all_test_probs.append(test_probs)

print('CV Macro F1:', np.mean(fold_scores))

# =====================================================
# 8) SOFT VOTING + SUBMISSION
# =====================================================
mean_probs = np.mean(all_test_probs, axis=0)
preds = np.argmax(mean_probs, axis=1)

submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': preds
})
submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(submission.head())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.0 MB/s eta 0:00:00
Device: cuda


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fold 5 Epoch 3 Macro F1: 0.82858


Fold 5 Epoch 4: 100%|██████████| 567/567 [04:56<00:00,  1.91it/s]


Fold 5 Epoch 4 Macro F1: 0.83359


Fold 5 Epoch 5: 100%|██████████| 567/567 [04:56<00:00,  1.91it/s]


Fold 5 Epoch 5 Macro F1: 0.83422
CV Macro F1: 0.8407148784697268
submission.csv saved!
   id  sentiment
0   0          0
1   1          1
2   2          2
3   3          2
4   4          0
